In [1]:
import os
from scipy.signal import spectrogram 
from scipy.io import wavfile
import numpy as np
from PIL import Image
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt

In [2]:
audio_folder=r'C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other'
output_folder='4_spec'
os.makedirs(output_folder, exist_ok=True)

In [3]:
# We need to create parmeters for the spectrogram
freq_min=10
freq_max=1000
window_size=1024
overlapp=800
n_fft=1024*4  #2^10

In [4]:
for file in os.listdir(audio_folder):
    file_path= os.path.join(audio_folder,file)
    print(file_path)
    fs,x=wavfile.read(file_path)
    f,t,S=spectrogram(x,fs,nperseg=window_size,noverlap=overlapp,nfft=n_fft)
    f_mask=(f>=freq_min)&(f<=freq_max)
    sxx=S[f_mask,:]
    G=10*np.log10(sxx+1e-8)
    G=np.flipud(G)
    # normalize the spectrogram
    norm=Normalize(vmin=np.min(G),vmax=np.max(G))
    G_normalized=norm(G)
    # convert spectrogram to image
    G_colormap=plt.cm.jet( G_normalized)
    G_image=( G_colormap[:,:,:3]*255).astype(np.uint8)
    G_resized=Image.fromarray(G_image).resize((224,224))
    output_file=os.path.join(output_folder,os.path.basename(file_path).replace('.wav','.png'))
    G_resized.save(  output_file)
    
        
    

C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__10.wav
C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__11.wav
C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__12.wav
C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__13.wav
C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__14.wav
C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__15.wav
C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__16.wav
C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__17.wav
C:\Users\yusle\OneDrive\Desktop\boat_audio\audio\UUV\UUV数据集\other\20220624110853_S_No7_F_W_0_label__0__18.wav
C:\Users\y

KeyboardInterrupt: 

In [3]:
# create model for training
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm

In [4]:
# prepare data
data_dir = r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\SplitClasses"  # Replace with your folder path
# Hyperparameters
batch_size = 8
num_epochs = 10
learning_rate = 0.001
num_classes =4  # Number of subfolders
print(num_classes)

# Image transformations
transform = transforms.Compose([transforms.ToTensor()])

4


In [ ]:
#for splitting folder with subfolder classes into folder with train and validation 70/30 split folder and each having all the subfolder classes  
import os
import shutil
import random

# ===== CONFIG =====
source_dir = r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\classes"   # folder with class subfolders
train_dir = r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\SplitClasses\Train"
test_dir = r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\SplitClasses\Val"
split_ratio = 0.7
random_seed = 42
# ==================

random.seed(random_seed)

# Create Train and Test directories
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Loop over each class folder
for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    # Create class folders in Train and Test
    train_class_dir = os.path.join(train_dir, class_name)
    test_class_dir = os.path.join(test_dir, class_name)

    os.makedirs(train_class_dir, exist_ok=True)
    os.makedirs(test_class_dir, exist_ok=True)

    # Get all image files
    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    random.shuffle(images)

    split_index = int(len(images) * split_ratio)
    train_images = images[:split_index]
    test_images = images[split_index:]

    # Copy files
    for img in train_images:
        shutil.copy2(
            os.path.join(class_path, img),
            os.path.join(train_class_dir, img)
        )

    for img in test_images:
        shutil.copy2(
            os.path.join(class_path, img),
            os.path.join(test_class_dir, img)
        )

    print(f"{class_name}: {len(train_images)} train / {len(test_images)} test")

print("✅ Dataset split complete.")


1_spec: 1866 train / 801 test
2_spec: 1 train / 1 test
3_spec: 4383 train / 1879 test
4_spec: 1400 train / 600 test
✅ Dataset split complete.


In [5]:
# Load dataset
train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "Train"), transform=transform)
val_dataset = datasets.ImageFolder(root=os.path.join(data_dir, "Val"), transform=transform)

In [6]:

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [12]:
class ViTModel(nn.Module):
    def __init__(self, num_classes):
        super(ViTModel, self).__init__()
        # Load pretrained Vision Transformer model
        self.model = models.vit_b_16(pretrained=True)
        
        # Replace the head (classification layer)
        in_features = self.model.heads.head.in_features  # Get input features of the head
        self.model.heads.head = nn.Linear(in_features, num_classes)  # Replace with new classification layer

    def forward(self, x):
        return self.model(x)

In [7]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

True
1
NVIDIA GeForce RTX 3070


In [11]:
#declaring the efficentNetModel structure 
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetModel, self).__init__()
        self.model = models.efficientnet_b7(pretrained=True)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [10]:
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.efficientnet_b0(pretrained=True)

        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.model(x)


In [30]:
#declaring the efficentNetModel b3 structure 
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetModel, self).__init__()
        self.model = models.efficientnet_b3(pretrained=True)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [13]:
# Initialize models, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

vit_model = ViTModel(num_classes).to(device)
efficientnet_model = EfficientNetModel(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
vit_optimizer = optim.Adam(vit_model.parameters(), lr=learning_rate)
efficientnet_optimizer = optim.Adam(efficientnet_model.parameters(), lr=learning_rate)

cuda


c:\Users\yusle\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\yusle\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
c:\Users\yusle\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B7_W

In [14]:
def train_model(model, optimizer, train_loader, val_loader, num_epochs, model_name):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Training {model_name} Epoch {epoch+1}/{num_epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader.dataset)
        accuracy = correct / total * 100

        print(f"{model_name} Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Accuracy: {accuracy:.2f}%")


In [51]:

# Train Vision Transformer
train_model(vit_model, vit_optimizer, train_loader, val_loader, num_epochs, "ViT")

Training ViT Epoch 1/1: 100%|██████████| 957/957 [49:41<00:00,  3.12s/it]


ViT Epoch 1/1, Train Loss: 1.0102, Val Loss: 0.9945, Val Accuracy: 57.27%


In [15]:
# Train EfficientNet
train_model(efficientnet_model, efficientnet_optimizer, train_loader, val_loader, num_epochs, "EfficientNet")


Training EfficientNet Epoch 1/10: 100%|██████████| 957/957 [03:11<00:00,  4.99it/s]


EfficientNet Epoch 1/10, Train Loss: 0.9246, Val Loss: 0.7996, Val Accuracy: 61.87%


Training EfficientNet Epoch 2/10: 100%|██████████| 957/957 [02:57<00:00,  5.38it/s]


EfficientNet Epoch 2/10, Train Loss: 0.7788, Val Loss: 0.7373, Val Accuracy: 66.35%


Training EfficientNet Epoch 3/10: 100%|██████████| 957/957 [02:59<00:00,  5.32it/s]


...

EfficientNet Epoch 8/10, Train Loss: 0.5029, Val Loss: 0.5230, Val Accuracy: 74.22%


Training EfficientNet Epoch 9/10: 100%|██████████| 957/957 [03:09<00:00,  5.04it/s]


EfficientNet Epoch 9/10, Train Loss: 0.4938, Val Loss: 0.5097, Val Accuracy: 76.32%


Training EfficientNet Epoch 10/10: 100%|██████████| 957/957 [03:00<00:00,  5.29it/s]


EfficientNet Epoch 10/10, Train Loss: 0.4610, Val Loss: 0.5368, Val Accuracy: 73.51%


In [13]:
# Save only the model weights
torch.save(efficientnet_model.state_dict(), 'trained_model_weights.pt')

NameError: name 'efficientnet_model' is not defined

In [ ]:
#loading saved model later
# Recreate the model architecture (must know the model achitecture I presume)
model2 = Net()

# Load the saved weights
model2.load_state_dict(torch.load('trained_model_weights.pt'))
model2.eval()  # important: switch to evaluation mode


fix code from here down 

In [7]:
from torchvision import datasets
from torch.utils.data import DataLoader

val_dataset = datasets.ImageFolder(
    root=r"C:\Users\yusle\OneDrive\Desktop\Git Hub Repositories\OSINModels2\SplitClasses\Val",
    transform=transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

class_names = val_dataset.classes
num_classes = len(class_names)


NameError: name 'transform' is not defined

In [8]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score


In [9]:
def evaluate_model(model, dataloader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Accuracy
    accuracy = accuracy_score(all_labels, all_preds)

    # Recall (macro)
    recall = recall_score(all_labels, all_preds, average="macro")

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    # Sensitivity & Specificity
    sensitivity = []
    specificity = []

    for i in range(len(cm)):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)

        sens = TP / (TP + FN) if (TP + FN) > 0 else 0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0

        sensitivity.append(sens)
        specificity.append(spec)

    results = {
        "accuracy": accuracy,
        "recall_macro": recall,
        "sensitivity_per_class": sensitivity,
        "specificity_per_class": specificity,
        "sensitivity_macro": np.mean(sensitivity),
        "specificity_macro": np.mean(specificity),
        "confusion_matrix": cm
    }

    return results


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

metrics = evaluate_model(model, val_loader, device)


NameError: name 'model' is not defined

In [ ]:
print(f"Accuracy:    {metrics['accuracy']:.4f}")
print(f"Recall:      {metrics['recall_macro']:.4f}")
print(f"Sensitivity: {metrics['sensitivity_macro']:.4f}")
print(f"Specificity: {metrics['specificity_macro']:.4f}")

print("\nPer-class metrics:")
for i, class_name in enumerate(class_names):
    print(
        f"{class_name:10s} | "
        f"Sensitivity: {metrics['sensitivity_per_class'][i]:.4f} | "
        f"Specificity: {metrics['specificity_per_class'][i]:.4f}"
    )
